In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

GETTING DATA

In [ ]:
from pathlib import Path

cwd = Path.cwd()
data_folder = Path.cwd().parent / 'data_preprocessing' / 'processed_data'
train_data = data_folder / 'train_ready.parquet'

try:
    if not train_data.is_file():
        raise ValueError()
except Exception as e:
    print('file with .parquet not found. Please check in "ai/ai_tools/dataprocessing/processed_data".')
    print('Ensure that file under processed_data direcory is called *EXACTLY* "train_ready.parquet".')
    print('Else check path in this cell for path mistypes.')

SPLITTING

In [ ]:
df = pd.read_parquet(train_data)

X = df['question_body']
Y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    Y , 
                                                    test_size=0.2, 
                                                    random_state=42)



SETTING UP TEXT EMBEDDOR

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

embeddor = TfidfVectorizer(stop_words='english')

LOADING CANDIDATE MODELS

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# some of these models take a while to train on CPU, so we choose models which can train relatively quickly
# and are known performers
Models = {
    'LinearSVC': Pipeline([('text_vectorizer', embeddor), ('model', LinearSVC(dual=False))]),
    'KNeighborsClassifier': Pipeline([('text_vectorizer', embeddor), ('model', KNeighborsClassifier(n_neighbors=5))]),
    'DecisionTreeClassifier': Pipeline([('text_vectorizer', embeddor), ('model', DecisionTreeClassifier(max_depth=20))]),
}

RUN TRAINING ON ALL MODELS

In [ ]:
import time
import gc
from sklearn.metrics import classification_report, accuracy_score

def train_models(models : dict):
    
    training_times = {}
    inference_times = {}
    accuracies = {}
    f1_scores = {}
    precision_scores = {}
    
    for model_name, model in models.items():
    
        # train model, get training time
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        #inference time
        start_time = time.time()
        y_pred = model.predict(X_test)
        inference_time = time.time() - start_time
        
        # accuracy
        accuracy = accuracy_score(y_test, y_pred)
        
        # classification report
        class_report_dict = classification_report(y_test, y_pred, output_dict=True)
        
        # f1 score
        f1_avg_score = class_report_dict['weighted avg']['f1-score']
        precision = class_report_dict['weighted avg']['precision']
        
        # recording evaluation metrics
        training_times[model_name] = training_time
        inference_times[model_name] = inference_time
        accuracies[model_name] = accuracy
        f1_scores[model_name] = f1_avg_score
        precision_scores[model_name] = precision

        
        # printing evaluations
        print(f"Model: {model_name}")
        print(f"Training Time: {training_time:.6f} seconds")
        print(f"Prediction Time: {inference_time:.6f} seconds")
        print(f"Accuracy: {accuracy:.6f}")
        print("Classification Report:")
        print(classification_report(y_test, y_pred))
        print("="*60)
        
        # cleaning every model
        del model
        gc.collect()
        
        # pandas df of all classification reports for visualization
    results_df = pd.DataFrame({
    "Model": list(Models.keys()),
    "Accuracy": list(accuracies.values()),
    "Prediction Time (s)": list(inference_times.values()),
    "Training Time (s)": list(training_times.values()),
    "f1_scores" : list(f1_scores.values()),
    "precision_scores" : list(precision_scores.values())
    })
        
    return results_df


results_df = train_models(Models)

RESULT ANALYSIS

In [ ]:
results_df.sort_values(by=['Accuracy'], ascending=False)

WINNER GRID SEARCH TRAINING OPTIMIZATION

In [ ]:
import joblib
from sklearn.model_selection import GridSearchCV

save_path = Path.cwd() / 'linearsvc_pipeline.joblib'

if not save_path.exists():
    
    # AI to train
    pipeline = Pipeline([
        ('text_vectorizer', TfidfVectorizer(stop_words="english",
                                            max_features=15000,
                                            min_df=3)), 
        ('model', LinearSVC(dual=False, 
                            class_weight='balanced')) # balance the influence of each class to address imbalance
    
    ])

    # 2. Define the Parameter Grid (using the step_name__parameter syntax)
    param_grid = {
        
        # LinearSVC Settings
        'model__C': [0.1, 1, 10],                      # regularization
        'model__tol': [1e-4, 1e-3]                     # Tolerance for stopping criteria
    }

    # grid search set up
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='f1_macro',   
        cv=2,                 # 5 cross validations
        n_jobs=-1,            
        verbose=3 
    )

    # train
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_

    print("Best Parameters:", grid_search.best_params_)
    print("Best Score:", grid_search.best_score_)

    # Save the model using joblib
    joblib.dump(best_model, save_path,compress=9)
    print(f"Model saved to {save_path}")

else:
    print("Model already exists at this path.")

FUNCTIONALITY TESTING

In [ ]:
grid_text_model = joblib.load(save_path)

In [ ]:
    
topic_maps = {key : value for key, value in enumerate(df['topic'].value_counts().index)}

def predicts(df, text_classifier, topic_maps, questions):
    
    for question in questions:
        topic_predict = text_classifier.predict([question])[0].item()
        predict2text = topic_maps[topic_predict]
        
        print(f"Question: {question} -> Label: '{predict2text}'")
        
        
        
predicts(df=df,
        text_classifier=grid_text_model,
        topic_maps=topic_maps,
        questions=["What is an array?", # basic functionality testing
                   "How can I design my classes better?", # vagueness
                   "Who is donald trump?",# random test
                   "" # null
                   ] 
        )

CONFUSION MATRIX DISPLAY

In [ ]:
predictions = grid_text_model.predict(X_test)
true_values = y_test

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

classes = [topic_maps[index] for index in range(len(topic_maps))]
cm = confusion_matrix(true_values, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

# GEMINI GENERATED CODE: 

# 1. Create a much larger figure and axis before plotting
fig, ax = plt.subplots(figsize=(16, 16)) 

# 2. Pass the axis to the plot and rotate the x-ticks
disp.plot(ax=ax, xticks_rotation='vertical') 

# 3. Add tight_layout so the long labels don't get cut off the edge of the screen
plt.tight_layout() 
plt.show()